<a href="https://colab.research.google.com/github/Anushkakumbhar223/Image-Smoothing-Comparison-Tool/blob/main/Image_Smoothing_Gradio_Tool.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import cv2
import numpy as np
import gradio as gr

In [2]:

# 1. Add Salt-and-Pepper Noise

def add_salt_pepper_noise(image, amount=0.05):

    noisy = image.copy()

    height, width = image.shape[:2]
    total_pixels = height * width
    number_of_pixels = int(amount * total_pixels)

    # Salt noise
    salt_y = np.random.randint(
        0, height, number_of_pixels // 2
    )
    salt_x = np.random.randint(
        0, width, number_of_pixels // 2
    )

    noisy[salt_y, salt_x] = 255

    # Pepper noise
    pepper_y = np.random.randint(
        0, height, number_of_pixels // 2
    )
    pepper_x = np.random.randint(
        0, width, number_of_pixels // 2
    )

    noisy[pepper_y, pepper_x] = 0

    return noisy


In [3]:
# 2. Calculate Mean Squared Error (MSE)

def calculate_mse(original, result):

    original = original.astype(np.float32)
    result = result.astype(np.float32)

    mse = np.mean((original - result) ** 2)

    return round(float(mse), 2)


In [4]:
# 3. Calculate PSNR

def calculate_psnr(mse):

    if mse == 0:
        return float("inf")

    psnr = 10 * np.log10((255 ** 2) / mse)

    return round(float(psnr), 2)

In [5]:
# 4. Main Image Processing Function
def process_image(image, add_noise, noise_amount):
    # Check image
    if image is None:
        return (
            None, None, None, None,
            "N/A", "N/A", "N/A", "N/A", "N/A", "N/A",
            "Please upload an image."
        )

    # Gradio gives RGB NumPy image
    original = np.array(image).astype(np.uint8)

    # Add noise if selected
    if add_noise:
        working_image = add_salt_pepper_noise(original, float(noise_amount))
    else:
        working_image = original.copy()

    # Apply Filters
    mean_image = cv2.blur(working_image, (5, 5))
    gaussian_image = cv2.GaussianBlur(working_image, (5, 5), 0)
    median_image = cv2.medianBlur(working_image, 5)

    # Calculate Metrics
    mean_mse = calculate_mse(original, mean_image)
    mean_psnr = calculate_psnr(mean_mse)

    gauss_mse = calculate_mse(original, gaussian_image)
    gauss_psnr = calculate_psnr(gauss_mse)

    median_mse = calculate_mse(original, median_image)
    median_psnr = calculate_psnr(median_mse)

    return (
        working_image,
        mean_image,
        gaussian_image,
        median_image,
        f"{mean_mse}", f"{mean_psnr} dB",
        f"{gauss_mse}", f"{gauss_psnr} dB",
        f"{median_mse}", f"{median_psnr} dB",
        "Success"
    )

In [6]:
# 5. Create and Launch Gradio Interface
with gr.Blocks() as demo:
    gr.Markdown("# Image Denoising Filter Comparison")

    with gr.Row():
        input_img = gr.Image(label="Upload Original Image", type="numpy")
        with gr.Column():
            add_noise_checkbox = gr.Checkbox(label="Add Salt & Pepper Noise", value=True)
            noise_amount_slider = gr.Slider(minimum=0.01, maximum=0.3, value=0.05, step=0.01, label="Noise Amount")
            submit_btn = gr.Button("Process Image")

    with gr.Row():
        noisy_output = gr.Image(label="Noisy / Working Image")
        mean_output = gr.Image(label="Mean Filter (5x5)")
        gaussian_output = gr.Image(label="Gaussian Filter (5x5)")
        median_output = gr.Image(label="Median Filter (5x5)")

    with gr.Row():
        with gr.Column():
            gr.Markdown("### Mean Filter Stats")
            mean_mse_txt = gr.Textbox(label="MSE")
            mean_psnr_txt = gr.Textbox(label="PSNR")
        with gr.Column():
            gr.Markdown("### Gaussian Filter Stats")
            gauss_mse_txt = gr.Textbox(label="MSE")
            gauss_psnr_txt = gr.Textbox(label="PSNR")
        with gr.Column():
            gr.Markdown("### Median Filter Stats")
            median_mse_txt = gr.Textbox(label="MSE")
            median_psnr_txt = gr.Textbox(label="PSNR")

    status_txt = gr.Textbox(label="Status")

    submit_btn.click(
        fn=process_image,
        inputs=[input_img, add_noise_checkbox, noise_amount_slider],
        outputs=[
            noisy_output, mean_output, gaussian_output, median_output,
            mean_mse_txt, mean_psnr_txt,
            gauss_mse_txt, gauss_psnr_txt,
            median_mse_txt, median_psnr_txt,
            status_txt
        ]
    )

demo.launch(inline=True, share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2a146d3aafce868713.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
